# Advanced GRPO Training for SME Liquidity

This notebook runs a judge-friendly, notebook-first pipeline for the real liquidity environment:

1. tiny TRL run
2. tiny Unsloth run
3. hacking inspection
4. curriculum-aware stabilization
5. larger TRL run
6. export + before/after demo


## 1. Setup

The default profile is T4-safe: it uses `Qwen/Qwen3-0.6B` for the main TRL path and keeps the tiny Unsloth check lightweight.


In [ ]:
%pip install -q "trl==0.29.0" "transformers>=4.56.0" "accelerate>=1.0.0" "peft>=0.17.0" bitsandbytes sentencepiece
# Optional tiny Unsloth phase:
# %pip install -q unsloth


In [ ]:
from pathlib import Path

RUN_PROFILE = "submission"
MODEL_NAME = "Qwen/Qwen3-0.6B"
UNSLOTH_MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
TASK_NAME = "liquidity-correlation-hard"
DIFFICULTY = "hard"
TOTAL_PERIODS = 2
OUTPUT_DIR = Path("outputs/grpo_sme_liquidity_advanced")

PROFILE_DEFAULTS = {
    "tiny": {
        "eval_seeds": [1000, 1001],
        "inspection_seeds": [1000, 1001],
        "eval_max_steps": 10,
        "tiny_trl": {"num_samples": 8, "max_steps": 4, "max_episode_steps": 8, "learning_rate": 5e-6},
        "tiny_unsloth": {"num_samples": 8, "build_preference_dataset": False},
        "main_trl": {"num_samples": 12, "max_steps": 8, "max_episode_steps": 10, "learning_rate": 5e-6},
    },
    "stabilize": {
        "eval_seeds": [1000, 1001, 1002],
        "inspection_seeds": [1000, 1001, 1002],
        "eval_max_steps": 12,
        "tiny_trl": {"num_samples": 10, "max_steps": 6, "max_episode_steps": 10, "learning_rate": 5e-6},
        "tiny_unsloth": {"num_samples": 10, "build_preference_dataset": False},
        "main_trl": {"num_samples": 20, "max_steps": 16, "max_episode_steps": 12, "learning_rate": 5e-6},
    },
    "submission": {
        "eval_seeds": [1000, 1001, 1002, 1003],
        "inspection_seeds": [1000, 1001, 1002, 1003],
        "eval_max_steps": 12,
        "tiny_trl": {"num_samples": 12, "max_steps": 6, "max_episode_steps": 10, "learning_rate": 5e-6},
        "tiny_unsloth": {"num_samples": 12, "build_preference_dataset": False},
        "main_trl": {"num_samples": 32, "max_steps": 25, "max_episode_steps": 12, "learning_rate": 5e-6},
    },
}

if RUN_PROFILE not in PROFILE_DEFAULTS:
    raise ValueError(f"Unsupported RUN_PROFILE: {RUN_PROFILE!r}")

EVAL_SEEDS = PROFILE_DEFAULTS[RUN_PROFILE]["eval_seeds"]
INSPECTION_SEEDS = PROFILE_DEFAULTS[RUN_PROFILE]["inspection_seeds"]
EVAL_MAX_STEPS = PROFILE_DEFAULTS[RUN_PROFILE]["eval_max_steps"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{
    "RUN_PROFILE": RUN_PROFILE,
    "MODEL_NAME": MODEL_NAME,
    "UNSLOTH_MODEL_NAME": UNSLOTH_MODEL_NAME,
    "TASK_NAME": TASK_NAME,
    "DIFFICULTY": DIFFICULTY,
    "TOTAL_PERIODS": TOTAL_PERIODS,
    "OUTPUT_DIR": str(OUTPUT_DIR.resolve()),
    "EVAL_SEEDS": EVAL_SEEDS,
}


## 2. Environment Overview

This notebook uses the real SME liquidity environment, not a static dataset shortcut. The policy must output one canonical JSON action per step, stay inside the tool contract, and improve verifiable reward without exploiting the environment surface.


In [ ]:
from rl.demo import (
    evaluate_before_after_policies,
    evaluate_policy_checkpoints,
    save_run_manifest,
    save_training_dashboard,
)
from rl.train_grpo_advanced import (
    build_advanced_run_plan,
    decide_curriculum_adjustment,
    run_backend_phase,
)
from rl.train_grpo_trl import make_training_args, run_training_session

advanced_plan = build_advanced_run_plan(
    profile=RUN_PROFILE,
    output_dir=str(OUTPUT_DIR),
    model_name=MODEL_NAME,
    unsloth_model_name=UNSLOTH_MODEL_NAME,
    task_name=TASK_NAME,
    difficulty=DIFFICULTY,
    total_periods=TOTAL_PERIODS,
)
advanced_plan


## 3. Tiny TRL Run

Phase 5 starts small. We run a tiny explicit-rollout TRL experiment first and save real artifacts.


In [ ]:
tiny_trl = run_backend_phase(
    backend="trl",
    output_dir=OUTPUT_DIR / "tiny_trl",
    model_name=MODEL_NAME,
    task_name=TASK_NAME,
    difficulty=DIFFICULTY,
    total_periods=TOTAL_PERIODS,
    eval_seeds=EVAL_SEEDS,
    inspection_seeds=INSPECTION_SEEDS,
    eval_max_steps=EVAL_MAX_STEPS,
    arg_overrides=PROFILE_DEFAULTS[RUN_PROFILE]["tiny_trl"],
)
tiny_trl


## 4. Tiny Unsloth Run

We run a tiny Unsloth experiment next so judges can see that the pipeline was checked across both backends before scaling up.


In [ ]:
tiny_unsloth = run_backend_phase(
    backend="unsloth",
    output_dir=OUTPUT_DIR / "tiny_unsloth",
    model_name=UNSLOTH_MODEL_NAME,
    task_name=TASK_NAME,
    difficulty=DIFFICULTY,
    total_periods=TOTAL_PERIODS,
    eval_seeds=EVAL_SEEDS,
    inspection_seeds=INSPECTION_SEEDS,
    eval_max_steps=EVAL_MAX_STEPS,
    arg_overrides=PROFILE_DEFAULTS[RUN_PROFILE]["tiny_unsloth"],
)
tiny_unsloth


## 5. Tiny-Run Comparison

Instead of eyeballing logs, compare both tiny checkpoints on the same fixed seed panel.


In [ ]:
tiny_checkpoint_paths = {}
if tiny_trl.get("status") == "ok":
    tiny_checkpoint_paths["tiny_trl"] = tiny_trl["checkpoint_path"]
if tiny_unsloth.get("status") == "ok":
    tiny_checkpoint_paths["tiny_unsloth"] = tiny_unsloth["checkpoint_path"]

tiny_comparison = evaluate_policy_checkpoints(
    output_dir=str(OUTPUT_DIR),
    seeds=EVAL_SEEDS,
    total_periods=TOTAL_PERIODS,
    task_name=TASK_NAME,
    difficulty=DIFFICULTY,
    checkpoint_paths=tiny_checkpoint_paths,
    model_name=MODEL_NAME,
    max_steps=EVAL_MAX_STEPS,
    include_base=True,
    include_heuristic=True,
    artifact_prefix="tiny_policy",
)
tiny_comparison


## 6. Hacking Inspection

Phase 6 checks for malformed JSON, suspicious shortcuts, unsupported actions, repeated tool abuse, and low-diversity completions.


In [ ]:
tiny_trl_inspection = tiny_trl.get("inspection", {})
tiny_unsloth_inspection = tiny_unsloth.get("inspection", {})

{
    "tiny_trl_inspection_path": tiny_trl_inspection.get("inspection_path"),
    "tiny_trl_warnings": tiny_trl_inspection.get("inspection", {}).get("warnings", []),
    "tiny_unsloth_inspection_path": tiny_unsloth_inspection.get("inspection_path"),
    "tiny_unsloth_warnings": tiny_unsloth_inspection.get("inspection", {}).get("warnings", []),
}


## 7. Curriculum Decision

Phase 7 simplifies the main run only if the tiny trained run collapses into near-zero reward, high default rate, or high timeout rate.


In [ ]:
tiny_summary = None
if tiny_trl.get("status") == "ok":
    tiny_summary = tiny_trl["before_after"]["summary"]
elif tiny_unsloth.get("status") == "ok":
    tiny_summary = tiny_unsloth["before_after"]["summary"]

curriculum_decision = decide_curriculum_adjustment(
    tiny_summary=tiny_summary,
    difficulty=DIFFICULTY,
    total_periods=TOTAL_PERIODS,
)
curriculum_decision


## 8. Main TRL Training

Phase 8 scales up only after the tiny loop is stable enough to justify a longer run.


In [ ]:
main_args = make_training_args(
    model_name=MODEL_NAME,
    task_name=TASK_NAME,
    difficulty=curriculum_decision["difficulty"],
    total_periods=curriculum_decision["total_periods"],
    output_dir=str(OUTPUT_DIR),
    **PROFILE_DEFAULTS[RUN_PROFILE]["main_trl"],
)
main_training = run_training_session(main_args)
dashboard = save_training_dashboard(main_training["trainer"], output_dir=str(OUTPUT_DIR))
checkpoint_path = str(Path(main_training["checkpoint_path"]).resolve())
print(f"Final checkpoint path: {checkpoint_path}")
print(f"Reward curve path: {dashboard['reward_curve_path']}")
print(f"Training dashboard path: {dashboard['training_dashboard_path']}")


## 9. Before/After Evaluation

Phase 9 demonstrates improvement. Shared helper runs `policy="base"` and `policy="trained"` on the same fixed seed panel, with an optional heuristic reference transcript.


In [ ]:
final_evaluation = evaluate_before_after_policies(
    output_dir=str(OUTPUT_DIR),
    seeds=EVAL_SEEDS,
    total_periods=curriculum_decision["total_periods"],
    task_name=TASK_NAME,
    difficulty=curriculum_decision["difficulty"],
    checkpoint_path=checkpoint_path,
    model_name=MODEL_NAME,
    max_steps=EVAL_MAX_STEPS,
    include_heuristic=True,
)
print(f"Eval summary path: {final_evaluation['eval_summary_path']}")
print(f"Policy comparison path: {final_evaluation['policy_comparison_path']}")
print(f"Submission ready: {final_evaluation['summary']['metadata']['submission_ready']}")
final_evaluation['summary']


## 10. Exported Artifacts

This final cell writes a single manifest and prints absolute paths for the checkpoint and judge-facing artifacts.


In [ ]:
manifest = {
    "run_profile": RUN_PROFILE,
    "advanced_plan": advanced_plan,
    "tiny_trl": tiny_trl,
    "tiny_unsloth": tiny_unsloth,
    "tiny_comparison": tiny_comparison,
    "curriculum_decision": curriculum_decision,
    "main_training": {
        "checkpoint_path": checkpoint_path,
        "training_dashboard_path": dashboard["training_dashboard_path"],
        "reward_curve_path": dashboard["reward_curve_path"],
    },
    "final_evaluation": final_evaluation,
}
manifest_path = save_run_manifest(manifest, output_dir=str(OUTPUT_DIR))

print(f"Final checkpoint path: {checkpoint_path}")
print(f"Reward curve path: {dashboard['reward_curve_path']}")
print(f"Training dashboard path: {dashboard['training_dashboard_path']}")
print(f"Eval summary path: {final_evaluation['eval_summary_path']}")
print(f"Policy comparison path: {final_evaluation['policy_comparison_path']}")
print(f"Run manifest path: {manifest_path}")
print(f"Tiny TRL inspection path: {tiny_trl_inspection.get('inspection_path')}")
print(f"Tiny Unsloth inspection path: {tiny_unsloth_inspection.get('inspection_path')}")
